### Thêm feature

In [2]:
import pandas as pd
from utils import *
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV

In [44]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24'] / 24)
df = df.sort_values('Time')
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0,5.014760,0,1,0,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0,1.305626,0,1,0,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0,5.939276,0,1,0,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0,4.824306,0,1,0,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0,4.262539,0,1,0,0.0,1.0,1.0,0,1


In [45]:
features = df.drop(['Time','Class','Amount','Hour_from_start_mod24'], axis=1).columns.tolist()
target = "Class"

In [48]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)
pos, neg = int((y_train==1).sum()), int((y_train==0).sum())
xg_model = XGBClassifier(
    n_estimators=600,
    max_depth=17,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda = 0.8,
    gamma=0.8,
    min_child_weight=9,
    tree_method = "hist",
    scale_pos_weight=neg/max(pos, 1),
    eval_metric='aucpr',
    random_state=SEED
)

xg_model.fit(X_train, y_train)

p_val_xg  = xg_model.predict_proba(X_val)[:, 1]
p_test_xg = xg_model.predict_proba(X_test)[:,1]

rs_xg_val = log_eval(y_val, p_val_xg)
rs_xg_test = log_eval(y_test, p_test_xg)
print("Validation: ")
print(rs_xg_val)
print("Test:")
print(rs_xg_test)

X_train: (181584, 36) y_train: (181584,)
X_val: (45396, 36) y_val: (45396,)
X_test: (56746, 36) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391
Validation: 
{'threshold': 0.017, 'Cost': 2430.0, 'ROC_AUC': 0.9784417241192118, 'PR_AUC': 0.778323039666108, 'debiased_ece': 0.0003406847300427499, 'adaptive_ece': 0.0003584588759365735, 'Brier': 0.0004812460974790156}
Test:
{'threshold': 0.011, 'Cost': 2920.0, 'ROC_AUC': 0.9812429418407679, 'PR_AUC': 0.80501283930724, 'debiased_ece': 0.00023348581322391038, 'adaptive_ece': 0.00014870838868847114, 'Brier': 0.0004400592588353902}


### Calibrate

In [50]:
tscv = TimeSeriesSplit(n_splits=10)

In [51]:
cal_xg = CalibratedClassifierCV(
    estimator=xg_model,
    method='isotonic',
    cv=tscv
)

cal_xg.fit(X_val, y_val)

p_test_cal_xg = cal_xg.predict_proba(X_test)[:, 1]

rs_cal_xg_test = log_eval(y_test, p_test_cal_xg)
print("Test:")
print(rs_cal_xg_test)

Test:
{'threshold': 0.009000000000000001, 'Cost': 3295.0, 'ROC_AUC': 0.9679716948738688, 'PR_AUC': 0.769785604392212, 'debiased_ece': 0.00023515070797371307, 'adaptive_ece': 0.00019089358667335608, 'Brier': 0.00045013757820271886}
